In [ ]:
!pip install -q kagglehub pycountry

import os, re, glob, time
import numpy as np
import pandas as pd

START_YEAR = None   # None = no restriction on release year. Set e.g. 2006 to filter again.

In [ ]:
try:
    import kagglehub
    folder = kagglehub.dataset_download("shivamb/amazon-prime-movies-and-tv-shows")
    csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]
    print("Loaded from Kaggle:", csv_path)
except Exception as e:
    print("Kaggle download failed:", e)
    print("Download amazon_prime_titles.csv manually from Kaggle and upload it below.")
    from google.colab import files
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]

prime = pd.read_csv(csv_path)
print("Raw shape:", prime.shape)
print(prime.columns.tolist())

In [ ]:
prime = prime.rename(columns={
    "type": "type",
    "title": "title",
    "director": "director",
    "cast": "cast",
    "country": "production_country",   # where it was made, NOT where it streams
    "release_year": "release_year",
    "rating": "maturity_rating",       # PG / R / 13+ etc., NOT the IMDb rating
    "duration": "duration_raw",
    "listed_in": "genre",
})

prime["release_year"] = pd.to_numeric(prime["release_year"], errors="coerce")

# Optional filter, only applied if you set START_YEAR above
if START_YEAR is not None:
    prime = prime[prime["release_year"] >= START_YEAR].copy()

dur = prime["duration_raw"].astype(str)
prime["duration_minutes"] = pd.to_numeric(dur.str.extract(r"(\d+)\s*min", expand=False), errors="coerce")
prime["seasons"] = pd.to_numeric(dur.str.extract(r"(\d+)\s*Season", expand=False), errors="coerce")

for c in ["title", "director", "cast", "production_country", "genre", "type"]:
    prime[c] = prime[c].astype("string").str.strip()

print("Rows:", prime.shape[0])
print("Release year range:", int(prime["release_year"].min()), "to", int(prime["release_year"].max()))
prime.head()

In [ ]:
BASICS_URL  = "https://datasets.imdbws.com/title.basics.tsv.gz"
RATINGS_URL = "https://datasets.imdbws.com/title.ratings.tsv.gz"

wanted_types = {"movie", "tvMovie", "tvSeries", "tvMiniSeries"}

chunks = []
for chunk in pd.read_csv(
    BASICS_URL, sep="\t", na_values="\\N", dtype=str,
    usecols=["tconst", "titleType", "primaryTitle", "originalTitle", "startYear"],
    chunksize=1_000_000,
):
    chunks.append(chunk[chunk["titleType"].isin(wanted_types)])

basics = pd.concat(chunks, ignore_index=True)
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")

if START_YEAR is not None:
    basics = basics[basics["startYear"] >= START_YEAR]

ratings = pd.read_csv(RATINGS_URL, sep="\t", na_values="\\N")
imdb = basics.merge(ratings, on="tconst", how="left")

print("IMDb titles loaded:", imdb.shape)

In [ ]:
def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

# Make Prime's type line up with IMDb's types
prime["_kind"] = prime["type"].map({"Movie": "movie", "TV Show": "tv"})
imdb["_kind"] = imdb["titleType"].map({
    "movie": "movie", "tvMovie": "movie",
    "tvSeries": "tv", "tvMiniSeries": "tv",
})

prime["_key"] = prime["title"].map(norm) + "|" + prime["release_year"].astype("Int64").astype(str) + "|" + prime["_kind"].astype(str)

# Several IMDb entries can share a name+year; keep the one with the most votes
imdb = imdb.sort_values("numVotes", ascending=False)

def build_lookup(title_col):
    tmp = imdb.copy()
    tmp["_key"] = tmp[title_col].map(norm) + "|" + tmp["startYear"].astype("Int64").astype(str) + "|" + tmp["_kind"].astype(str)
    tmp = tmp.drop_duplicates("_key")
    return tmp[["_key", "tconst", "averageRating", "numVotes"]]

# Pass 1: match on primaryTitle
merged = prime.merge(build_lookup("primaryTitle"), on="_key", how="left")

# Pass 2: for still-unmatched rows, try originalTitle
missing = merged["averageRating"].isna()
second = prime[missing.values][["show_id", "_key"]].merge(build_lookup("originalTitle"), on="_key", how="left")
second = second.dropna(subset=["averageRating"]).drop_duplicates("show_id").set_index("show_id")

merged = merged.set_index("show_id")
for col in ["tconst", "averageRating", "numVotes"]:
    merged.loc[second.index, col] = second[col]
merged = merged.reset_index()

merged = merged.rename(columns={"averageRating": "imdb_rating", "numVotes": "imdb_votes"})

match_rate = merged["imdb_rating"].notna().mean() * 100
print(f"IMDb rating matched for {merged['imdb_rating'].notna().sum()} of {len(merged)} titles ({match_rate:.1f}%)")

In [ ]:
import requests, pycountry
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
import logging
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

TMDB_KEY = getpass("Paste your TMDB API key: ")
TEST_LIMIT = 3000  # test run. Set to None for the full catalog.
MAX_WORKERS = 10
BASE = "https://api.themoviedb.org/3"
ERROR = "__ERROR__"
session = requests.Session()
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session.mount("https://", adapter)

def tmdb_get(path, **params):
    params["api_key"] = TMDB_KEY
    for attempt in range(5):
        try:
            r = session.get(BASE + path, params=params, timeout=20)
        except requests.RequestException:
            time.sleep(2)
            continue
        if r.status_code == 429:            # rate limited, wait and retry
            time.sleep(2 * (attempt + 1))
            continue
        if r.status_code == 200:
            return r.json()
        return None                          # 404 etc. = not found
    return ERROR

def resolve_tmdb(row):
    kind = "movie" if row["type"] == "Movie" else "tv"
    key = "movie_results" if kind == "movie" else "tv_results"

    # Best match: via the IMDb ID found in Cell 5
    if pd.notna(row["tconst"]):
        j = tmdb_get(f"/find/{row['tconst']}", external_source="imdb_id")
        if j == ERROR:
            return kind, None, "api_error"
        if j and j.get(key):
            return kind, j[key][0]["id"], "imdb_id"

    # Fallback: search by title and year (less reliable)
    year = int(row["release_year"]) if pd.notna(row["release_year"]) else None
    params = {"query": row["title"]}
    if kind == "movie":
        params["year"] = year
    else:
        params["first_air_date_year"] = year
    j = tmdb_get(f"/search/{kind}", **params)
    if j == ERROR:
        return kind, None, "api_error"
    if j and j.get("results"):
        return kind, j["results"][0]["id"], "title_search"
    return kind, None, "not_found_on_tmdb"

def process(row):
    kind, tid, method = resolve_tmdb(row)
    if tid is None:
        return row["show_id"], None, method, "not_checked", []
    j = tmdb_get(f"/{kind}/{tid}/watch/providers")
    if j == ERROR:
        return row["show_id"], tid, method, "api_error", []
    found = []
    for cc, d in (j or {}).get("results", {}).items():
        for p in d.get("flatrate", []):      # flatrate = included with subscription
            if "prime video" in p["provider_name"].lower():
                found.append((cc, p["provider_name"]))
    status = "available" if found else "no_prime_flatrate_found"
    return row["show_id"], tid, method, status, found

todo = merged if TEST_LIMIT is None else merged.head(TEST_LIMIT)
rows = todo[["show_id", "type", "title", "release_year", "tconst"]].to_dict("records")

status_rows, avail_rows = [], []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process, r) for r in rows]
    for i, f in enumerate(as_completed(futures), 1):
        sid, tid, method, status, found = f.result()
        status_rows.append({"show_id": sid, "tmdb_id": tid,
                            "match_method": method, "availability_status": status})
        for cc, prov in found:
            avail_rows.append({"show_id": sid, "country_code": cc, "provider": prov})
        if i % 500 == 0:
            print(f"{i}/{len(rows)} titles done")

def country_name(cc):
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc

availability = pd.DataFrame(avail_rows, columns=["show_id", "country_code", "provider"])
availability["country_name"] = availability["country_code"].map(country_name)
availability = availability.drop_duplicates(["show_id", "country_code"])
status = pd.DataFrame(status_rows)

print("\nAvailability status counts:")
print(status["availability_status"].value_counts())
print("\nMatch method counts:")
print(status["match_method"].value_counts())
print("\nCountries with at least one title:", availability["country_code"].nunique())

In [ ]:
final = merged[[
    "show_id", "type", "title", "director", "cast", "production_country",
    "release_year", "duration_raw", "duration_minutes", "seasons",
    "genre", "maturity_rating", "imdb_rating", "imdb_votes",
]].copy()

final = final.merge(status, on="show_id", how="left")   # adds tmdb_id, match_method, availability_status

movies = final[final["type"] == "Movie"][["show_id", "title"]].rename(columns={"title": "film_title"})
tv_shows = final[final["type"] == "TV Show"][["show_id", "title"]].rename(columns={"title": "tv_show_name"})

def explode_col(df, col, newname):
    out = df[["show_id", col]].dropna().copy()
    out[col] = out[col].str.split(",")
    out = out.explode(col)
    out[col] = out[col].str.strip()
    out = out[out[col] != ""]
    return out.rename(columns={col: newname}).drop_duplicates()

cast_tbl     = explode_col(final, "cast", "cast_member")
genre_tbl    = explode_col(final, "genre", "genre_name")
prod_country_tbl = explode_col(final, "production_country", "production_country_name")
director_tbl = explode_col(final, "director", "director_name")

# Titles available per country, split by Movie / TV Show (for a map visual)
avail_typed = availability.merge(final[["show_id", "type"]], on="show_id", how="left")
country_summary = (
    avail_typed.groupby(["country_code", "country_name", "type"])
    .size().unstack(fill_value=0).rename_axis(columns=None).reset_index()
)
for col in ["Movie", "TV Show"]:
    if col not in country_summary.columns:
        country_summary[col] = 0
country_summary["total_titles"] = country_summary["Movie"] + country_summary["TV Show"]
country_summary = country_summary.sort_values("total_titles", ascending=False)

print("Release years in data:", int(final["release_year"].min()), "to", int(final["release_year"].max()))
print("\nMissing values per column (%):")
print((final.isna().mean() * 100).round(1).sort_values(ascending=False))
print("\nRows:", len(final), "| Movies:", len(movies), "| TV shows:", len(tv_shows))
print("Availability rows:", len(availability))

country_summary.head(15)

In [ ]:
os.makedirs("prime_export", exist_ok=True)

final.to_csv("prime_export/amazon_prime_main.csv", index=False)
movies.to_csv("prime_export/amazon_prime_movies.csv", index=False)
tv_shows.to_csv("prime_export/amazon_prime_tv_shows.csv", index=False)
cast_tbl.to_csv("prime_export/bridge_cast.csv", index=False)
genre_tbl.to_csv("prime_export/bridge_genre.csv", index=False)
prod_country_tbl.to_csv("prime_export/bridge_production_country.csv", index=False)
director_tbl.to_csv("prime_export/bridge_director.csv", index=False)
availability.to_csv("prime_export/bridge_availability_country.csv", index=False)
country_summary.to_csv("prime_export/country_availability_summary.csv", index=False)

!zip -q -r prime_export.zip prime_export

from google.colab import files
files.download("prime_export.zip")

In [ ]:
import requests
from getpass import getpass

key = getpass("TMDB v3 API key: ")
r = requests.get("https://api.themoviedb.org/3/movie/550/watch/providers", params={"api_key": key})
print(r.status_code)
print(r.text[:200])